In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.6 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188757 sha256=4511355bb6e0229c1041912a704cfcf7541bfeba3abd7211e558394a709e33a6
  Stored in directory: /root/.cache/pip/wheels/37/56/51/f27c4eb4bcee969fb4577a5b7d90f9cfe596ebd0e4def65ba5
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61447 sha256=9640230cbc6e62c3c18d2dc70805105f8969e4e57384faaaad207ec393e8299b
  Stored in directory: /root/.cache/pip/whe

In [2]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
# Connect to Google Drive
drive.mount('/content/drive', force_remount=True)

CLASSES = ['ClassOne', 'ClassTwo']
NUM_FRAMES = 16  # X3D defaults to 16 frames
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- THE MAGIC THRESHOLD ---
# Anything below 55% confidence gets sorted into Miscellaneous
THRESHOLD = 0.55

# The Paths
# Updated to match the X3D model name we saved during training
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/Trained_Models/g01_General_VideoClassifier_TwoClass_X3D_TrainedModel.pth"
TEST_FOLDER_PATH = "/content/drive/MyDrive/Test/Input_For_Sorting"
RESULT_FOLDER_PATH = "/content/drive/MyDrive/Result"

# Mapping the guesses to your folders (Added the Miscellaneous bucket!)
BUCKET_NAMES = {
    'ClassOne': 'Predicted_ClassOne',
    'ClassTwo': 'Predicted_ClassTwo',
    'miscellaneous': 'Predicted_Miscellaneous'
}

# --- 3. BUILD THE ROBOT BODY & LOAD THE BRAIN ---
print("Building the robot (X3D-M) and loading the brain...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=False)

# Change the head so it knows we only have 2 classes
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# --- 4. VIDEO GLASSES (TRANSFORMS) ---
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0),
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]), # X3D specifics
            Resize((256, 256)) # X3D specifics
        ])
    )
])

# --- 5. THE SORTING LOOP ---
print(f"\n--- STARTING THE SORTING MACHINE (Threshold: {THRESHOLD*100}%) ---")

# Look at every single file inside the test folder
for filename in os.listdir(TEST_FOLDER_PATH):

    # Only look at video files, ignore random hidden files
    if not filename.lower().endswith(('.mp4', '.avi', '.mov')):
        continue

    video_path = os.path.join(TEST_FOLDER_PATH, filename)
    print(f"\nWatching: {filename}...")

    try:
        # Load the 2-second clip and put the glasses on it
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)
        video_data = video_transform(video_data)
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        # Make the guess!
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # --- THE MISCELLANEOUS INTERCEPT ---
        confidence_score = conf.item()
        original_guess = CLASSES[pred_idx.item()]

        if confidence_score < THRESHOLD:
            winning_guess = 'miscellaneous'
            print(f"Guess: MISCELLANEOUS (Unsure about {original_guess.upper()}, only {confidence_score*100:.2f}% sure)")
        else:
            winning_guess = original_guess
            print(f"Guess: {winning_guess.upper()} ({confidence_score*100:.2f}% sure)")

        # --- THE FILE MOVING MAGIC ---
        # Look up the exact folder name from your dictionary
        target_bucket = BUCKET_NAMES[winning_guess]

        # Build the paths
        target_folder = os.path.join(RESULT_FOLDER_PATH, target_bucket)
        target_path = os.path.join(target_folder, filename)

        # ⭐️ IMPORTANT: Make sure the folder exists!
        # If 'Predicted_Miscellaneous' isn't in your drive yet, this line creates it automatically.
        os.makedirs(target_folder, exist_ok=True)

        # Copy the video into your folder!
        shutil.copy(video_path, target_path)
        print(f"-> Safely copied into {target_bucket}!")

    except Exception as e:
        print(f"Oh no! Something went wrong with {filename}: {e}")

print("\n--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---")

/usr/local/lib/python3.13/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
Guess: CLASSTWO (73.09% sure)
-> Safely copied into Predicted_ClassTwo!

Watching: apr18_t0000_f002975.mp4...
Guess: MISCELLANEOUS (Unsure about CLASSONE, only 52.76% sure)
-> Safely copied into Predicted_Miscellaneous!

Watching: apr17_t0009_f030525.mp4...
Guess: CLASSONE (72.77% sure)
-> Safely copied into Predicted_ClassOne!

Watching: apr17_t0009_f030914.mp4...
Guess: CLASSTWO (73.05% sure)
-> Safely copied into Predicted_ClassTwo!

Watching: apr17_t0009_f032909.mp4...
Guess: CLASSTWO (73.09% sure)
-> Safely copied into Predicted_ClassTwo!

Watching: apr17_t0009_f029331.mp4...
Guess: CLASSTWO (72.56% sure)
-> Safely copied into Predicted_ClassTwo!

Watching: apr17_t0009_f026720.mp4...
Guess: CLASSONE (69.44% sure)
-> Safely copied into Predicted_ClassOne!

Watching: apr18_t0000_f012982.mp4...
Guess: CLASSTWO (71.56% sure)
-> Safely copied into Predicted_ClassTwo!

Watching: apr17_t0009_f024813.mp4...
Guess: CLASSONE (69.38% sure)
-